<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/03_baseline_rf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03: Baseline Final con Análisis Multicriterio
**Proyecto:** Detección de Deslizamientos mediante Inteligencia Artificial  
**Institución:** Universidad EAFIT  

Este notebook integra el entrenamiento de Random Forest con métricas avanzadas: Importancia de Variables, Matriz de Confusión y Curva ROC, corregido para NumPy 2.0.

In [ ]:
from google.colab import drive
import os, h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

# 1. Montar Drive
drive.mount('/content/drive')

# 2. Rutas de datos
base_path = Path('/content/drive/MyDrive/Landslide4Sense')
img_dirs = list(base_path.glob('**/TrainData/img'))

if img_dirs:
    train_img_dir = img_dirs[0]
    train_mask_dir = train_img_dir.parent / "mask"
    img_list = sorted(list(train_img_dir.glob("*.h5")))
    mask_list = sorted(list(train_mask_dir.glob("*.h5")))
    print(f"✅ {len(img_list)} muestras detectadas.")
else:
    print("❌ ERROR: Verifica la ruta en Drive.")

In [ ]:
N_SAMPLES = 1000 
X, y = [], []

for i in tqdm(range(min(N_SAMPLES, len(img_list))), desc="Extrayendo características"):
    with h5py.File(img_list[i], "r") as hf:
        patch = hf[list(hf.keys())[0]][()]
    with h5py.File(mask_list[i], "r") as hf:
        mask = hf[list(hf.keys())[0]][()]
    
    # --- CORRECCIÓN NUMPY 2.0 ---
    rgb = patch[:,:,[3,2,1]]
    rgb_norm = ((rgb - np.min(rgb)) / (np.ptp(rgb) + 1e-8) * 255).astype("uint8")
    
    # Descriptores: HOG + Slope (Banda 13)
    feats_hog = hog(rgb_norm, pixels_per_cell=(16,16), cells_per_block=(2,2), channel_axis=-1, feature_vector=True)
    avg_slope = np.mean(patch[:,:,12])
    
    X.append(np.hstack([feats_hog, avg_slope]))
    y.append(int(np.max(mask) > 0))

X, y = np.array(X), np.array(y)
print(f"✓ Dataset listo: {X.shape}")

In [ ]:
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, jaccard_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
)
import json as json_lib

output_dir = Path('/content/drive/MyDrive/Landslide4Sense/results/random_forest')
output_dir.mkdir(parents=True, exist_ok=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_records = []
f1_scores, auc_scores, prec_scores, rec_scores, iou_scores = [], [], [], [], []

print('Entrenando Baseline Random Forest...')
for fold, (t_idx, v_idx) in enumerate(skf.split(X, y)):
    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    rf.fit(X[t_idx], y[t_idx])

    preds = rf.predict(X[v_idx])
    probs = rf.predict_proba(X[v_idx])[:, 1]

    f1   = f1_score(y[v_idx],   preds, zero_division=0)
    prec = precision_score(y[v_idx], preds, zero_division=0)
    rec  = recall_score(y[v_idx],   preds, zero_division=0)
    aucs = roc_auc_score(y[v_idx],  probs)
    iou  = jaccard_score(y[v_idx],  preds, zero_division=0)

    f1_scores.append(f1)
    auc_scores.append(aucs)
    prec_scores.append(prec)
    rec_scores.append(rec)
    iou_scores.append(iou)
    fold_records.append({
        'fold': fold+1, 'best_f1': f1,
        'precision': prec, 'recall': rec,
        'auc': aucs, 'iou': iou,
    })
    print(
        f'  Fold {fold+1}: F1={f1:.4f} | AUC={aucs:.4f} | '
        f'Prec={prec:.4f} | Rec={rec:.4f} | IoU={iou:.4f}'
    )

mean_f1  = float(np.mean(f1_scores))
std_f1   = float(np.std(f1_scores))
mean_auc = float(np.mean(auc_scores))

print(f'\nF1 medio : {mean_f1:.4f} +/- {std_f1:.4f}')
print(f'AUC medio: {mean_auc:.4f}')

# Version 0 — compatible con notebook 07
summary_v0 = {
    'model'  : 'random_forest',
    'version': 0,
    'config' : {'n_estimators': 100, 'n_samples': N_SAMPLES, 'n_folds': 5},
    'folds'  : [{'fold': r['fold'], 'best_f1': r['best_f1']} for r in fold_records],
    'mean_f1': mean_f1,
    'std_f1' : std_f1,
}
with open(output_dir / 'kfold_summary.json', 'w') as fout:
    json_lib.dump(summary_v0, fout, indent=2)
print('Guardado: kfold_summary.json (v0)')

# Version 2 — metricas completas
summary_v2 = {
    'model'    : 'random_forest',
    'version'  : 2,
    'config'   : {'n_estimators': 100, 'n_samples': N_SAMPLES, 'n_folds': 5},
    'folds'    : fold_records,
    'mean_f1'  : mean_f1,
    'std_f1'   : std_f1,
    'mean_auc' : mean_auc,
    'std_auc'  : float(np.std(auc_scores)),
    'mean_prec': float(np.mean(prec_scores)),
    'mean_rec' : float(np.mean(rec_scores)),
    'mean_iou' : float(np.mean(iou_scores)),
}
with open(output_dir / 'kfold_summary_v2.json', 'w') as fout:
    json_lib.dump(summary_v2, fout, indent=2)
print('Guardado: kfold_summary_v2.json (v2)')

# Visualizaciones
fig, ax = plt.subplots(1, 3, figsize=(22, 6))

importances = rf.feature_importances_
indices = np.argsort(importances)[-15:]
ax[0].barh(range(len(indices)), importances[indices], color='teal')
ax[0].set_yticks(range(len(indices)))
ax[0].set_yticklabels(
    ['HOG_' + str(i) if i < X.shape[1]-1 else 'PENDIENTE (B13)' for i in indices]
)
ax[0].set_title('Top 15 Importancia de Caracteristicas')

cm = confusion_matrix(y[v_idx], preds)
ConfusionMatrixDisplay(cm, display_labels=['Estable', 'Desliz.']).plot(
    cmap='YlGnBu', ax=ax[1], colorbar=False
)
ax[1].set_title('Matriz de Confusion (Ultimo Fold)')

fpr, tpr, _ = roc_curve(y[v_idx], probs)
roc_auc_val = auc(fpr, tpr)
ax[2].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc_val:.3f}')
ax[2].plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
ax[2].set_title('Curva ROC')
ax[2].legend(loc='lower right')

plt.suptitle('Random Forest Baseline — Resultados', fontsize=13)
plt.tight_layout()
plt.savefig(output_dir / 'rf_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nRESUMEN BASELINE RF')
print('-'*50)
print(f'F1 medio : {mean_f1:.4f} +/- {std_f1:.4f}')
print(f'AUC medio: {mean_auc:.4f}')
print(f'Precision: {float(np.mean(prec_scores)):.4f}')
print(f'Recall   : {float(np.mean(rec_scores)):.4f}')
print(f'IoU      : {float(np.mean(iou_scores)):.4f}')
print('-'*50)
